# Optimize LLM inference with openvino.genai cache eviction and sparse attention techniques

[openvino.genai](https://github.com/openvinotoolkit/openvino.genai) provides a set of LLM pipelines with familiar API and many effective optimizations for Intel(R) hardware-accelerated inference. 
In this notebook the [cache eviction](https://openvinotoolkit.github.io/openvino.genai/docs/concepts/optimization-techniques/kvcache-eviction-algorithm) and [sparse attention prefill](https://openvinotoolkit.github.io/openvino.genai/docs/concepts/optimization-techniques/sparse-attention-prefill) techniques will be demonstrated to achieve generation time and KV cache usage improvements throughout the entire generation process.


This example will demonstrate using RAG engines as a tool in an agent with OpenVINO and LlamaIndex.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download models](#Download-models)
- [Create openvino.genai pipeline](#Create-openvino.genai-pipeline)
- [Run generation](#Run-generation)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).


# Prerequisites

[back to top ⬆️](#Table-of-contents:)

Install required dependencies:

In [ ]:
!pip install transformers optimum-intel openvino openvino_genai openvino_tokenizers

# Download model

[back to top ⬆️](#Table-of-contents:)

The model will be downloaded from the huggingface model repositories and converted for optimized OpenVINO(TM) inference using wrappers from [optimum-intel](https://github.com/huggingface/optimum-intel) and [openvino.genai](https://github.com/openvinotoolkit/openvino.genai) Python APIs:

In [ ]:
from os.path import sep
from transformers import AutoTokenizer, AutoModelForCausalLM
from optimum.intel import OVModelForCausalLM
from openvino_tokenizers import convert_tokenizer
import openvino as ov
from openvino import save_model
import openvino.properties.hint as hints
import pathlib

model_id = "Qwen/Qwen2-0.5B-Instruct"  # or any other supported LLM model id from HF
models_path = pathlib.Path(str(model_id).replace(sep, "_"))
hf_tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, local_files_only=False)
llm_props = {
          hints.inference_precision: ov.Type.f32,
          hints.kv_cache_precision: ov.Type.f16,
      }
opt_model = OVModelForCausalLM.from_pretrained(model_id, export=True, compile=False, load_in_8bit=False, trust_remote_code=True, ov_config=llm_props, local_files_only=False)
opt_model.save_pretrained(models_path)
opt_model.config.save_pretrained(models_path)

hf_tokenizer.save_pretrained(models_path)
tokenizer, detokenizer = convert_tokenizer(hf_tokenizer, with_detokenizer=True)
      
save_model(tokenizer, models_path / "openvino_tokenizer.xml")
save_model(detokenizer, models_path / "openvino_detokenizer.xml")

# Create openvino.genai pipeline
openvino.genai pipelines provide a `.generate` method with huggingface-aligned API. 
To configure the pipeline to utilize cache eviction and sparse prefill, the corresponding fields in `openvino_genai.SchedulerConfig` should be set to `True` as demonstrated below.

Optionally, the respective configuration structs could be tuned for the specifics of your particular use case.
Refer to the documentation ([1](https://openvinotoolkit.github.io/openvino.genai/docs/concepts/optimization-techniques/kvcache-eviction-algorithm), [2](https://openvinotoolkit.github.io/openvino.genai/docs/concepts/optimization-techniques/sparse-attention-prefill)) to learn about the implementation and available options.

In [ ]:
import openvino_genai

scheduler_config = openvino_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 2000  # total cache available
scheduler_config.use_cache_eviction = True  # works best for long generation lengths
scheduler_config.use_sparse_attention = True  # works best for long prompts

# If necessary, configure the cache eviction and sparse attention parameters manually, e.g.:
# scheduler_config.cache_eviction_config = openvino_genai.CacheEvictionConfig(start_size = 32, 
#                                                                             recent_size = 32, 
#                                                                             max_cache_size = 128,
#                                                                             aggregation_mode = openvino_genai.AggregationMode.NORM_SUM)
# 
# scheduler_config.sparse_attention_config.num_last_dense_tokens_in_prefill = 32
# scheduler_config.sparse_attention_config.num_retained_start_tokens_in_cache = 32
# scheduler_config.sparse_attention_config.num_retained_recent_tokens_in_cache = 32

device = "CPU" # or "GPU"
model_cb = openvino_genai.ContinuousBatchingPipeline(models_path, scheduler_config, device, {}, llm_props)

# Run generation

Using `openvino_genai.GenerationConfig`, we specify greedy sampling and the number of tokens we want to generate. 
The `.generate` method allows specifying separate generation configs for each prompt - in our case there is only one prompt to process.

After the `.generate` call the generated text, the wall clock time elapsed and cache usage metrics are printed out. Compare these between the cases with `.use_cache_eviction = True` vs `.use_cache_eviction = False`, and `.use_sparse_attention = True` vs `.use_sparse_attention = False`. 
Best optimization ratios are achieved for longer prompts and longer generation lengths.

In [ ]:
generation_config = openvino_genai.GenerationConfig()  # greedy sampling by default
generation_config.num_return_sequences = 1
generation_config.max_new_tokens = 5000
generation_config.ignore_eos = True # to generate exactly generation_config.max_new_tokens without EOS token cut-off 

import requests
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"}
wiki_text = requests.get('https://en.wikipedia.org/wiki/Elvis_Presley', headers=headers).text

prompt = "Rewrite the following HTML page without HTML tags:\n\n" + wiki_text + "\nRewritten text:\n"


import time
start = time.time()
ans_batch = model_cb.generate([prompt], [generation_config])
end = time.time()

print(ans_batch[0].m_generation_ids[0])

metrics = model_cb.get_metrics()
print(f"Wall time: {(end - start) :.2f} s")
print(f"Max cache usage: {metrics.max_cache_usage:3.2f}%")
print(f"Avg cache usage: {metrics.avg_cache_usage:3.2f}%")
